In [ ]:
"""File này để test xem thiết kế json vậy có khả quan hay không"""

In [ ]:
json_mock = [
    {
        "code": "A00.0",
        "text": ["text content","text content","text content"],
        "retriever": {
            "sapbert": {
                "001": 2,
                "034": 2,
                "012": 2,
            },
            "rapidfuzz": {
                "001": 2,
                "001": 2,
                "001": 2,
            },
        }
    }
]

In [2]:
for record in json_mock:
    code = record["code"]
    text = record["text"]

    print(f"Code: {code}")
    print(f"Text: {text}")

    # Loop through retrievers: sapbert, rapidfuzz
    for retriever_name, candidates in record["retriever"].items():
        print(f"\n{retriever_name} results:")

        # Loop through each candidate code and its score
        for candidate_code, score in candidates.items():
            print(f"  Candidate: {candidate_code}, score: {score}")

Code: A00.0
Text: text content

sapbert results:
  Candidate: 001, score: 2
  Candidate: 034, score: 2
  Candidate: 012, score: 2

rapidfuzz results:
  Candidate: 001, score: 2


In [2]:
from rapidfuzz import process, fuzz
import numpy as np

def rapidfuzz_retrieval(text_list, candidate_texts, candidate_codes, top_k=5, scorer=fuzz.WRatio):
    score_matrix = process.cdist(text_list, candidate_texts, scorer=scorer, workers=-1)
    code_scores = {}
    top_idx = np.argsort(-score_matrix, axis=1)[:, :top_k]
    for row_i, row in enumerate(top_idx):
        for idx in row:
            code = candidate_codes[idx]
            score = score_matrix[row_i, idx]
            if code not in code_scores or score > code_scores[code]:
                code_scores[code] = round(float(score), 2)
    return code_scores


# Sample candidate database
candidate_texts = [
    "essential hypertension",
    "high blood pressure",
    "type 2 diabetes mellitus",
    "bacterial pneumonia",
    "acute kidney failure",
]

candidate_codes = [
    "I10",
    "I10",      # Same code, different synonym
    "E11.9",
    "J15.9",
    "N17.9",
]


# Query expansions for one medical entity
text_list = [
    "hypertension",
    "high blood pressure",
]


result = rapidfuzz_retrieval(
    text_list=text_list,
    candidate_texts=candidate_texts,
    candidate_codes=candidate_codes,
    top_k=3,
)

print(result)

# Basic checks
assert isinstance(result, dict)
assert "I10" in result
assert result["I10"] == 100.0
assert all(isinstance(score, float) for score in result.values())

print("Test passed!")

{'I10': 100.0, 'E11.9': 38.89, 'J15.9': 31.58}
Test passed!
